# pHash 近重复算子测试

自包含 smoke 验证入口，用于测试单个算子 `duplicate.perceptual_duplicate_check`。

In [ ]:
from pathlib import Path
import sys

# 从当前工作目录向上查找仓库根目录，保证 Notebook 从 notebooks/ 或仓库根目录启动都可用。
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("cannot locate repository root")

# 优先使用当前 checkout 的源码，避免 editable install 指向其他 worktree 或旧路径。
for path in (str(repo_root / "src"), str(repo_root)):
    if path in sys.path:
        sys.path.remove(path)
for path in reversed((str(repo_root / "src"), str(repo_root))):
    sys.path.insert(0, path)

print(f"repo_root={repo_root}")

In [ ]:
from PIL import Image, ImageDraw
import pandas as pd

from image_gallery.cleaning import BasicCleaner
from image_gallery.dataset import Dataset
from notebooks._helpers.paths import get_notebook_library_root, reset_output_dir

# 所有临时图片、raw parquet 和清洗产物都写到 Notebook 私有目录，重复运行前会自动清理。
library_root = reset_output_dir(get_notebook_library_root("operators_phash_duplicate"))
image_dir = library_root / "images"
image_dir.mkdir(parents=True, exist_ok=True)
raw_dataset_path = library_root / "raw.parquet"
cleaning_output_dir = library_root / "cleaning"

print(f"library_root={library_root}")

In [ ]:
def build_keeper_image() -> Image.Image:
    # 构造一张包含色块和斜线的基准图，让 pHash 有稳定的低频特征。
    image = Image.new("RGB", (96, 96), color=(120, 130, 140))
    draw = ImageDraw.Draw(image)
    draw.rectangle((20, 20, 76, 76), fill=(180, 60, 90))
    draw.line((0, 95, 95, 0), fill=(30, 200, 180), width=4)
    return image


keeper_path = image_dir / "keeper.png"
near_path = image_dir / "near.png"
far_path = image_dir / "far.png"
broken_path = image_dir / "broken.jpg"

# keeper 是近重复组中应该被保留的第一张图片。
keeper = build_keeper_image()
keeper.save(keeper_path)

# near 只改动少量像素：字节不同，但视觉几乎一致，默认 max_distance=4 下应被 drop。
near = build_keeper_image()
near.putpixel((4, 4), (121, 131, 141))
near.putpixel((5, 4), (122, 132, 142))
near.save(near_path)

# far 使用明显不同的构图，pHash 距离应超过阈值，最终保持 keep。
far = Image.new("RGB", (96, 96), color=(240, 240, 240))
draw = ImageDraw.Draw(far)
draw.ellipse((10, 10, 86, 86), fill=(10, 10, 10))
far.save(far_path)

# broken 用于验证坏图不会生成 phash，也不会误判为重复。
broken_path.write_bytes(b"not an image")

raw_frame = pd.DataFrame(
    {
        "image_id": ["keeper", "near", "far", "broken"],
        "image_uri": [str(keeper_path), str(near_path), str(far_path), str(broken_path)],
    }
)
dataset = Dataset.write(raw_frame, str(raw_dataset_path))
print(dataset.to_frame().to_string(index=False))

In [ ]:
# 只启用单个 pHash 近重复算子，避免其他清洗规则干扰验证结果。
operator_configs = [{"duplicate.perceptual_duplicate_check": {"max_distance": 4}}]
cleaner = BasicCleaner(operator_configs).compile()

# 编译计划必须先计算 phash，再做 dataset_aggregate 级别的近重复分组。
plan_frame = cleaner.plan()
print(plan_frame.to_string(index=False))
assert plan_frame["computer_name"].tolist() == [
    "image_perceptual_hash_computer",
    "perceptual_duplicate_group_computer",
]

# 执行后会写出 parameter_table、evaluation_table、state 和 relation table。
cleaner.run(dataset, output_dir=cleaning_output_dir)
run_dirs = sorted(path for path in cleaning_output_dir.iterdir() if path.is_dir())
assert len(run_dirs) == 1
run_dir = run_dirs[0]
print(f"run_dir={run_dir}")

In [ ]:
# 读取运行产物，分别检查参数、评估结果和近重复 pair relation。
parameter_table = pd.read_parquet(run_dir / "parameter_table.parquet")
evaluation_table = pd.read_parquet(run_dir / "evaluation_table.parquet")
relation_table = pd.read_parquet(run_dir / "relations" / "perceptual_duplicate_pairs.parquet")

print("parameter_table")
print(
    parameter_table[
        [
            "image_id",
            "phash",
            "perceptual_duplicate_group_id",
            "perceptual_duplicate_count",
            "perceptual_duplicate_distance",
        ]
    ].to_string(index=False)
)
print("evaluation_table")
print(
    evaluation_table[
        [
            "image_id",
            "perceptual_duplicate_action",
            "perceptual_duplicate_reason",
            "final_action",
        ]
    ].to_string(index=False)
)
print("relation_table")
print(relation_table.to_string(index=False))

In [ ]:
# 参数断言：near 与 keeper 的 phash 一致，far 不同，broken 为空。
parameter_rows = parameter_table.set_index("image_id")
evaluation_rows = evaluation_table.set_index("image_id")

assert parameter_rows.loc["keeper", "phash"] == parameter_rows.loc["near", "phash"]
assert parameter_rows.loc["keeper", "phash"] != parameter_rows.loc["far", "phash"]
assert parameter_rows.loc["broken", "phash"] == ""
assert int(parameter_rows.loc["near", "perceptual_duplicate_distance"]) <= 4
assert pd.isna(parameter_rows.loc["far", "perceptual_duplicate_distance"])

# 评估断言：每组第一张 keep，近重复成员 drop，非重复和坏图 keep。
assert evaluation_rows.loc["keeper", "perceptual_duplicate_action"] == "keep"
assert evaluation_rows.loc["near", "perceptual_duplicate_action"] == "drop"
assert evaluation_rows.loc["far", "perceptual_duplicate_action"] == "keep"
assert evaluation_rows.loc["broken", "perceptual_duplicate_action"] == "keep"
assert evaluation_rows.loc["near", "final_action"] == "drop"

# relation 断言：只记录 keeper -> near 这一组近重复关系。
assert len(relation_table) == 1
assert relation_table.loc[0, "relation_type"] == "perceptual_duplicate"
assert relation_table.loc[0, "source_image_id"] == "keeper"
assert relation_table.loc[0, "target_image_id"] == "near"

print("PASS: phash duplicate operator notebook smoke completed")